# System1Bench — Colab T4
从公开 fork 克隆并运行 Laya 评测，无需 GH_TOKEN。
先选择 T4 GPU，再从上到下执行。结果保存到 Colab 临时磁盘，完成后下载 ZIP。

复用 Colab 当前 Python 和预装 PyTorch，不下载或重装 PyTorch/CUDA。
创建可访问 Colab 系统包的虚拟环境，安装 Laya 0.3.20、Transformers 5.16.1 和项目依赖。
安装前检查依赖解析不会改动 torch；安装后执行依赖、模块和真实 GPU 推理检查。
本 notebook 尚未在 Colab 实机验证，T4 结果应与原始 A100 参考环境分开报告。


In [ ]:
# 1. 复用 Colab 当前 Python 和预装 PyTorch
import os, sys, json, subprocess
from pathlib import Path
from importlib.metadata import version

assert Path("/content").exists(), "请在 Google Colab 中运行"
assert sys.version_info >= (3, 12), "需要 Python 3.12 或更新版本"
LOG_DIR = Path("/content/system1bench-logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
COMMAND_NUMBER = 0

def run(args, cwd=None):
    global COMMAND_NUMBER
    COMMAND_NUMBER += 1
    args = list(map(str, args))
    log = LOG_DIR / f"command-{COMMAND_NUMBER:03d}.log"
    while log.exists():
        COMMAND_NUMBER += 1
        log = LOG_DIR / f"command-{COMMAND_NUMBER:03d}.log"
    print("+", " ".join(args), flush=True)
    print("日志:", log, flush=True)
    lines = []
    with log.open("w", encoding="utf-8") as stream:
        stream.write("+ " + " ".join(args) + "\n")
        stream.flush()
        proc = subprocess.Popen(
            args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", bufsize=1)
        try:
            for line in proc.stdout:
                print(line, end="", flush=True)
                stream.write(line)
                stream.flush()
                lines.append(line)
            code = proc.wait()
        except BaseException:
            proc.terminate()
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
            raise
    if code:
        tail = "".join(lines[-40:])
        raise RuntimeError(
            f"命令退出码 {code}；完整日志：{log}\n最后 40 行：\n{tail}")


run(["nvidia-smi"])
import torch
assert torch.cuda.is_available(), "请选择 GPU 运行时"
TORCH_VERSION = version("torch")
print("复用 PyTorch:", TORCH_VERSION, "GPU:", torch.cuda.get_device_name(0))

REPO = Path("/content/system1bench")
if not REPO.exists():
    run(["git", "clone", "--depth", "1",
         "https://github.com/Jaffe2718/system1bench.git", REPO])
assert (REPO / "requirements-colab-t4.txt").is_file(), "现有克隆过旧，请在新运行时重新克隆本 fork"
run(["git", "rev-parse", "HEAD"], cwd=REPO)

PY = Path("/content/system1bench-colab-venv/bin/python")
if not PY.exists():
    run([sys.executable, "-m", "venv", "--system-site-packages", PY.parent.parent])
CONSTRAINT = Path("/content/system1bench-torch-constraint.txt")
CONSTRAINT.write_text("torch==" + TORCH_VERSION)
run([PY, "--version"])


In [ ]:
# 2. 安装项目依赖；保留 Colab 预装 PyTorch
# 先 dry-run 审核解析计划，若要求改动 torch 则停止。
PLAN = Path("/content/system1bench-install-plan.json")
install = [PY, "-m", "pip", "install", "--constraint", CONSTRAINT,
           "-r", "requirements-colab-t4.txt"]
run(install + ["--dry-run", "--report", PLAN], cwd=REPO)
plan = json.loads(PLAN.read_text())
planned_names = {
    item["metadata"]["name"].lower().replace("_", "-")
    for item in plan.get("install", [])
}
assert "torch" not in planned_names, "依赖解析试图安装 torch，已停止，请保留日志排查"
run(install, cwd=REPO)
run([PY, "-c",
     "from importlib.metadata import version; import sys; "
     "assert version('torch') == sys.argv[1]; print('PyTorch preserved:', version('torch'))",
     TORCH_VERSION])
run([PY, "-m", "pip", "check"])
run([PY, "-m", "pip", "freeze"])


In [ ]:
# 3. 导入检查：覆盖数据下载、准备、模型评测和报告所需模块
run([PY, "-c", """
import importlib, sys
modules = [
    'numpy', 'pyarrow', 'pyarrow.parquet', 'huggingface_hub',
    'yaml', 'requests', 'torch', 'transformers', 'laya',
    'safetensors', 'tokenizers',
    'transformers.models.modernbert.modeling_modernbert',
    'system1bench.download', 'system1bench.prepare',
    'system1bench.laya_adapter', 'system1bench.run',
    'system1bench.metrics', 'system1bench.report',
]
for name in modules:
    importlib.import_module(name)
    print('IMPORT OK:', name, flush=True)
import torch, transformers
assert sys.version_info >= (3, 12)
assert torch.cuda.is_available(), 'CUDA 不可用，请选择 GPU 运行时'
print('GPU:', torch.cuda.get_device_name(0))
print('Compute capability:', torch.cuda.get_device_capability(0))
print('torch:', torch.__version__, 'transformers:', transformers.__version__)
"""], cwd=REPO)

In [ ]:
# 4. 模型与输出目录：只使用 Colab 临时磁盘
MODEL = "english"  # 可改为 "multilingual"
BATCH_SIZE = 1

assert MODEL in ("english", "multilingual")
OUTPUT = Path("/content/system1bench-results/laya-t4-bs1-v1")
OUTPUT.mkdir(parents=True, exist_ok=True)
print("模型:", MODEL, "结果目录:", OUTPUT)
print("临时磁盘不跨会话保留，请在会话结束前下载结果 ZIP。")

# 在推理前记录依赖和仓库提交，便于排错与复现。
freeze = subprocess.check_output([str(PY), "-m", "pip", "freeze"], text=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
(OUTPUT / f"environment-{MODEL}.txt").write_text(freeze)
(OUTPUT / f"repository-{MODEL}.txt").write_text(commit)


In [ ]:
# 5. 固定版本数据下载与准备
# 子进程日志会实时显示并写入 /content/system1bench-logs。
# 失败时查看异常中最后 40 行的实际原因，不要跳过哈希校验。
run([PY, "-u", "-m", "system1bench.download"], cwd=REPO)
run([PY, "-u", "-m", "system1bench.prepare"], cwd=REPO)


In [ ]:
# 6. 实际加载固定版本模型，检查三个决策原语
# 独立进程退出后释放显存；本单元格失败时先排错，不要继续全量评测。
smoke = r'''
import sys
from system1bench.laya_adapter import LayaAdapter
from system1bench.common import decode
a = LayaAdapter(sys.argv[1])
assert a.agent.device.type == "cuda", "模型回退到了 CPU"
print("AMP dtype:", a.agent.dtype, "enabled:", a.agent.amp_enabled, flush=True)
q = {
    "department": {
        "type": "choice",
        "instructions": "Which department should handle this request?",
        "criteria": {"billing": "payments and refunds", "technical": "technical problems"}
    },
    "refund": {"type": "noul", "instructions": "Does the user request a refund?"},
    "urgency": {
        "type": "score", "instructions": "How urgent is this request?",
        "criteria": ["not urgent", "soon", "urgent"]
    }
}
out = a.predict(
    [{"message": "I was charged twice. Please refund me today."}],
    q, "en", {"max_len": 8192, "head_max_len": 4096}, 1)
assert len(out) == 1
for qid, question in q.items():
    pred, probs = decode(question, out[0]["answers"][qid])
    assert pred is not None
    print("DECODE OK:", qid, pred, probs, flush=True)
print("GPU INFERENCE OK", flush=True)
'''
run([PY, '-u', '-c', smoke, MODEL], cwd=REPO)

下面开始正式评测：36 个套件，每模型 26,450 个决策。启动后可能需要较长时间。
进度以 DONE 行为准，并检查 errors。评测器会将部分推理异常保存为错误记录，不能只看进程退出成功。
中断后可重跑相同命令，按完整套件恢复；改变 batch、依赖、代码、模型版本或 GPU 后请使用新输出目录。
运行 multilingual 时修改上面的 MODEL，重新执行配置、推理检查及后续单元格；无需重装环境或重复下载数据。

In [ ]:
# 7. 正式评测
run([PY, "-u", "-m", "system1bench.run",
     "--model", MODEL, "--batch-size", str(BATCH_SIZE),
     "--output", OUTPUT], cwd=REPO)

# 检查该模型所有套件的错误计数
import json
meta = json.loads((OUTPUT / MODEL / "metadata.json").read_text())
assert meta["status"] == "DONE"
errors = {name: s["errors"] for name, s in meta["suites"].items() if s["errors"]}
print("有错误的套件:", errors)
assert not errors, "存在推理错误，请保留日志排查；不要将此结果视为完整成功"

In [ ]:
# 8. 汇总与报告（同一输出目录内的模型均应完成）
run([PY, "-m", "system1bench.metrics", "--results", OUTPUT], cwd=REPO)
run([PY, "-m", "system1bench.report", "--results", OUTPUT], cwd=REPO)

print("结果已保存到:", OUTPUT)
for name in ("summary.json", "metrics.csv", "RESULTS.zh-CN.md"):
    print(OUTPUT / name)

In [ ]:
# 9. 保存依赖版本和本次使用的仓库提交，打包下载
import shutil
from google.colab import files
freeze = subprocess.check_output(
    [str(PY), "-m", "pip", "freeze"], text=True)
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
(OUTPUT / "environment-pip-freeze.txt").write_text(freeze)
(OUTPUT / "repository-commit.txt").write_text(commit)
shutil.copytree(LOG_DIR, OUTPUT / "logs", dirs_exist_ok=True)
archive = shutil.make_archive("/content/system1bench-t4-results", "zip", OUTPUT)
files.download(archive)